In [ ]:
import pandas as pd
from imblearn.over_sampling import SMOTE
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

#read data
dataset = pd.read_csv(r'D:\1work\1 PhD\Demo_ecoli.csv')
X = dataset.drop('ecoli', axis=1)
X.fillna(0, inplace=True)
y = dataset['ecoli']
feature_names = X.columns.tolist()

# Stratified sampling is used to divide the training and test subsets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=79, stratify=y)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# SMOTE
smote = SMOTE(sampling_strategy='auto', random_state=6)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

X_train = pd.DataFrame(X_train_resampled, columns=feature_names)
y_train = y_train_resampled
X_test = pd.DataFrame(X_test, columns=feature_names)
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix
import xgboost as xgb

finxgb = xgb.XGBClassifier(n_estimators=75, learning_rate=0.042, max_depth=6, random_state=80,
                           gamma=0, subsample=0.1,
                           colsample_bytree=0.9, min_child_weight=6,
                           use_label_encoder=False, eval_metric='logloss')

finxgb.fit(X_train, y_train)
y_pred = finxgb.predict(X_test)
y_pred_proba = finxgb.predict_proba(X_test)[:, 1]
test_accuracy = accuracy_score(y_test, y_pred)
test_precision = precision_score(y_test, y_pred)
test_recall = recall_score(y_test, y_pred)
test_f1 = f1_score(y_test, y_pred)
test_roc_auc = roc_auc_score(y_test, y_pred_proba)
test_conf_matrix = confusion_matrix(y_test, y_pred)
test_sensitivity = test_recall
test_specificity = test_conf_matrix[0, 0] / (test_conf_matrix[0, 0] + test_conf_matrix[0, 1])
print("\nTest set performance:")
print(f"Accuracy: {test_accuracy:.4f}")
print(f"Sensitivity: {test_sensitivity:.4f}")
print(f"Specificity: {test_specificity:.4f}")
print(f"ROC-AUC: {test_roc_auc:.4f}")

In [ ]:
#Plot the Roc cruve
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc
finxgb.fit(X_train, y_train)
y_test_pred_proba = finxgb.predict_proba(X_test)[:, 1] 
plt.rcParams['font.family'] = 'Times New Roman'
plt.rcParams['font.size'] = 14
fpr_test, tpr_test, thresholds_test = roc_curve(y_test, y_test_pred_proba)
roc_auc_test = auc(fpr_test, tpr_test)
plt.figure(figsize=(5, 4))
plt.plot(fpr_test, tpr_test, color='darkorange', lw=2,
         label=f'ROC curve (area = {roc_auc_test:.2f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC)')
plt.legend(loc="lower right")
plt.grid(True)
plt.tight_layout()  
plt.show()

In [ ]:
#SHAP analysis
import shap
import matplotlib.pyplot as plt
explainer = shap.TreeExplainer(finxgb)
shap_values = explainer.shap_values(X_train)

if isinstance(shap_values, list):

    shap_values_class_1 = shap_values[1]
else:

    shap_values_class_1 = shap_values
fig, ax = plt.subplots(figsize=(3, 7))
shap.summary_plot(shap_values_class_1, X_train, feature_names=feature_names, max_display=24, show=False)
fig.set_size_inches(3, 7)
plt.subplots_adjust(left=0.1) 
plt.rcParams['font.family'] = 'Times New Roman'
plt.rcParams['font.size'] = 20

ax = plt.gca()
ax.tick_params(axis='y', labelsize=20)  
ax.set_xlim([-0.65, 0.5])
x_tick_labels = ax.get_xticklabels()
for label in x_tick_labels:
    label.set_fontsize(20)  

y_tick_labels = ax.get_yticklabels()
for label in y_tick_labels:
    label.set_fontsize(20) 
ax.set_xlabel('SHAP value', fontsize=20)
x_label = ax.set_xlabel(ax.get_xlabel(), fontsize=20)
y_label = ax.set_ylabel(ax.get_ylabel(), fontsize=20)
plt.show()

In [ ]:
#Relative contribution of 24 features
import shap
import numpy as np
import pandas as pd
explainer = shap.TreeExplainer(finxgb)
shap_values = explainer.shap_values(X_train)
if isinstance(shap_values, list):
    shap_values_class_1 = shap_values[1]
else:
    shap_values_class_1 = shap_values

mean_abs_shap = np.abs(shap_values_class_1).mean(axis=0)

relative_contrib = 100 * mean_abs_shap / mean_abs_shap.sum()
feature_importance_df = pd.DataFrame({
    'feature': feature_names,
    'mean_abs_shap': mean_abs_shap,
    'relative_contrib_%': relative_contrib
})

feature_importance_df = feature_importance_df.sort_values('relative_contrib_%', ascending=False)

print(feature_importance_df)

In [ ]:
#Find the cutoff
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix
from sklearn.metrics import roc_curve, precision_recall_curve
def calculate_sensitivity_specificity(y_true, y_proba, thresholds):
    sensitivity = []
    specificity = []
    for thresh in thresholds:
        y_pred_thresh = (y_proba >= thresh).astype(int)
        conf_matrix = confusion_matrix(y_true, y_pred_thresh)
        tn, fp, fn, tp = conf_matrix.ravel()
        sens = tp / (tp + fn) if (tp + fn) > 0 else 0
        spec = tn / (tn + fp) if (tn + fp) > 0 else 0
        sensitivity.append(sens)
        specificity.append(spec)
    return sensitivity, specificity
fpr, tpr, roc_thresholds = roc_curve(y_test, y_pred_proba)
thresholds = np.linspace(0, 1, 10000) 
sensitivity, specificity = calculate_sensitivity_specificity(y_test, y_pred_proba, thresholds)
thresholds = np.sort(np.unique(y_pred_proba))
sensitivity, specificity = calculate_sensitivity_specificity(y_test, y_pred_proba, thresholds)
sensitivity = np.array(sensitivity)
specificity = np.array(specificity)
diff = sensitivity - specificity
idx = np.where(diff[:-1]*diff[1:] < 0)[0]

if len(idx)==0:
    print("non")
else:
    i = idx[0]
    x0, x1 = thresholds[i], thresholds[i+1]
    y0, y1 = diff[i], diff[i+1]
    cross_threshold = x0 - y0 * (x1-x0) / (y1-y0)

    sens0, sens1 = sensitivity[i], sensitivity[i+1]
    spec0, spec1 = specificity[i], specificity[i+1]

    sens_cross = sens0 + (sens1-sens0)*(cross_threshold-x0)/(x1-x0)
    spec_cross = spec0 + (spec1-spec0)*(cross_threshold-x0)/(x1-x0)

    print(f"Intersection at threshold={cross_threshold:.6f}")
    print(f"Sensitivity = {sens_cross:.6f}, Specificity = {spec_cross:.6f}")

    plt.figure(figsize=(5, 4)) 

    plt.rcParams['font.family'] = 'Times New Roman'
    plt.rcParams['font.size'] = 14
    plt.plot(thresholds, sensitivity, label="Sensitivity", color="blue", linewidth=2)
    plt.plot(thresholds, specificity, label="Specificity", color="green", linewidth=2)
    plt.axvline(x=cross_threshold, color="red", linestyle="--", 
                label=f"Threshold: {cross_threshold:.4f}")
    plt.scatter([cross_threshold], [sens_cross], color='purple', s=100, zorder=5)
    plt.scatter([cross_threshold], [spec_cross], color='orange', s=100, zorder=5)
    plt.legend(fontsize=12)
    plt.xlabel("Threshold")
    plt.ylabel("Score")
    plt.tight_layout()  
    plt.show()

In [ ]:
#Predict the API exceedance in all riverine grid cells
import pandas as pd
import numpy as np
import xgboost as xgb
finxgb.fit(X_train, y_train)

predX = pd.read_csv(r'D:\1work\Demo_prediction.csv')
predX.fillna(0, inplace=True)
predX = scaler.transform(predX)
pred_prob = finxgb.predict_proba(predX)[:, 1]
best_threshold =0.4679
predY_custom = (pred_prob >= best_threshold).astype(int)
print("Using custom threshold:")
print(np.unique(predY_custom, return_counts=True))

result_df = pd.DataFrame({
    'Id': np.arange(1, len(predY_custom) + 1),
    'level': predY_custom,  
    'probably': pred_prob   
})
output_path = r'D:\1work\1 PhD\ecoli_custom_threshold.csv'
result_df.to_csv(output_path, index=False)

In [ ]:
#Map the API exceedance in global riverine grid cells
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

predY_df = pd.read_csv(r'D:\1work\1 PhD\ecoli_custom_threshold.csv')

shapefile_path = r'D:\1work\1 PhD\Gis test\globalriver2km.shp'
gdf = gpd.read_file(shapefile_path)

contient_shapefile_path = r'D:\1work\1 PhD\Gis test\contient.shp'
contient_gdf = gpd.read_file(contient_shapefile_path)

if 'probably' in gdf.columns:
    gdf = gdf.drop(columns=['probably'])

if 'Id' not in predY_df.columns or 'probably' not in predY_df.columns:
    raise ValueError("The CSV file must contain 'Id' and 'probably' columns.")

merged_gdf = gdf.merge(predY_df, on='Id', how='left')

if 'probably' not in merged_gdf.columns:
    raise ValueError("The merge operation failed. Ensure 'probably' column is present in the CSV file and matches the Ids in the Shapefile.")

gdf_green = merged_gdf[(merged_gdf['probably'] >= 0) & (merged_gdf['probably'] < 0.4679)]
gdf_yellow = merged_gdf[(merged_gdf['probably'] >= 0.4679) & (merged_gdf['probably'] < 0.5)]
gdf_orange = merged_gdf[(merged_gdf['probably'] >= 0.5) & (merged_gdf['probably'] < 0.75)]
gdf_red = merged_gdf[(merged_gdf['probably'] >= 0.75) & (merged_gdf['probably'] <= 1)]

fig, ax = plt.subplots(1, 1, figsize=(10, 12), dpi=400)

contient_gdf.plot(ax=ax, color='none', edgecolor='gray', linewidth=0.5, label='Continent')

if not gdf_green.empty:
    gdf_green.plot(ax=ax, color='#87CEFA', edgecolor='#87CEFA', linewidth=0.6, label='0-0.2424')

if not gdf_yellow.empty:
    gdf_yellow.plot(ax=ax, color='yellow', edgecolor='yellow', linewidth=0.6, label='0.2424-0.5')

if not gdf_orange.empty:
    gdf_orange.plot(ax=ax, color='orange', edgecolor='orange', linewidth=0.6, label='0.5-0.75')

if not gdf_red.empty:
    gdf_red.plot(ax=ax, color='red', edgecolor='red', linewidth=0.6, label='0.75-1')

plt.legend()
ax.set_aspect('equal')
plt.show()